In [62]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

def display_df(
    df, rows=pd.options.display.max_rows, cols=pd.options.display.max_columns
):
    """Displays n rows and cols from df"""
    with pd.option_context(
        "display.max_rows", rows, "display.max_columns", cols
    ):
        display(df)

# Example Notebook – Practical Data Science
This notebook demonstrates core text analysis concepts such as term frequency, inverse document frequency, and their combination in TF–IDF.


### Agenda 📆

- Text features.
- Bag of words.
- Cosine similarity.
- TF-IDF.
    - Example: State of the Union addresses 🎤.

## Text features

Suppose we'd like to predict the **sentiment** of a piece of text from 1 to 10.
- 10: Very positive (happy).
- 1: Very negative (sad, angry).

Example:
- Input: "CS 1 is a pretty good class."
- Output: 7.

- We can frame this as a regression problem, but we can't directly use the inputs, because they are **text**, not **numbers**.

### Text features

- **Big question: How do we represent a text document as a feature vector of numbers?**

- If we can do this, we can:
    - use a text document as input in a regression or classification model (in a few lectures).
    - **quantify** the similarity of two text documents (today).

### Example: San Diego employee salaries

- [Transparent California](https://transparentcalifornia.com/salaries/san-diego/) publishes the salaries of all City of San Diego employees.
- Let's look at the 2022 data.

In [63]:
salaries = pd.read_csv('https://transcal.s3.amazonaws.com/public/export/san-diego-2022.csv')
salaries['Employee Name'] = salaries['Employee Name'].str.split().str[0] + ' Xxxx'

In [64]:
salaries.head()

,Employee Name,Job Title,Base Pay,Overtime Pay,Other Pay,Benefits,Total Pay,Pension Debt,Total Pay & Benefits,Year,Notes,Agency,Status
0,Mara Xxxx,City Attorney,227441.53,0.00,-3686.44,100767.53,223755.09,307892.43,632415.04,2022,NaN,San Diego,FT
1,Todd Xxxx,Mayor,227441.53,0.00,9409.20,79705.90,236850.73,307892.43,624449.05,2022,NaN,San Diego,FT
2,Terence Xxxx,Assistant Police Chief,227224.32,0.00,39949.52,60174.08,267173.84,197708.00,525055.93,2022,NaN,San Diego,FT
3,Esmeralda Xxxx,Police Sergeant,124604.40,162506.54,52893.66,38934.40,340004.60,143797.64,522736.64,2022,NaN,San Diego,FT
4,Marcelle Xxxx,Assistant Retirement Administrator,279868.04,0.00,-5989.14,74720.15,273878.90,171613.74,520212.79,2022,NaN,San Diego,FT


### Goal: Quantifying similarity
- Our goal is to describe, numerically, how similar two job titles are.
- For example, 'Deputy Fire Chief' should be more similar to 'Fire Battalion Chief' than to 'City Attorney'.
- Idea: Two job titles are similar if they share words, regardless of order. We can measure this by counting shared words.

### Canonicalization
Before counting shared words, we first make job titles consistent:

- Some titles include punctuation like '-' or '&', which should be ignored so 'Assistant - Manager' and 'Assistant Manager' count as the same.
- Some include "glue" words like 'to' or 'the', which can also be ignored.
- Titles with roman numerals, like 'Police Officer II' and 'Police Officer I', should be treated as the same.

This process of standardizing titles is called canonicalization.


### Punctuation

Are there job titles with extra punctuation we can remove?

- We can check using a regular expression that finds any character not a letter, number, or space.

In [65]:
jobtitles = salaries['Job Title']
jobtitles.head()

0                         City Attorney
1                                 Mayor
2                Assistant Police Chief
3                       Police Sergeant
4    Assistant Retirement Administrator
Name: Job Title, dtype: object

In [66]:
# Uses character class negation.
jobtitles.str.contains(r'[^A-Za-z0-9 ]', regex=True).sum()

np.int64(922)

In [67]:
jobtitles[jobtitles.str.contains(r'[^A-Za-z0-9 ]', regex=True)].head()

137          Park & Recreation Director
248     Associate Engineer - Mechanical
734     Associate Engineer - Electrical
882        Associate Engineer - Traffic
1045         Associate Engineer - Civil
Name: Job Title, dtype: object

It seems like we should replace these pieces of punctuation with a single space.

### "Glue" words

Are there job titles with "glue" words in the middle, such as `'Assistant to the Manager'`?

To figure out if any titles contain the word `'to'`, we **can't** just do the following, because it will evaluate to `True` for job titles that have `'to'` anywhere in them, even if not as a standalone word.

In [68]:
# Why are we converting to lowercase?
jobtitles.str.lower().str.contains('to').sum()

np.int64(1577)

In [69]:
jobtitles[jobtitles.str.lower().str.contains('to')]

0                             City Attorney
4        Assistant Retirement Administrator
8                  Retirement Administrator
17                     Deputy City Attorney
21                      Department Director
                        ...                
12751                        Test Monitor I
12768         Recreation Center Director Ii
12778                        Test Monitor I
12812                           Custodian I
12826              Word Processing Operator
Name: Job Title, Length: 1577, dtype: object

Instead, we need to look for `'to'` separated by word boundaries.

In [70]:
jobtitles.str.lower().str.contains(r'\bto\b', regex=True).sum()

np.int64(10)

In [71]:
jobtitles[jobtitles.str.lower().str.contains(r'\bto\b', regex=True)]

1638              Assistant To The Chief Operating Officer
2183                  Principal Assistant To City Attorney
2238                             Assistant To The Director
4845               Confidential Secretary To City Attorney
5159                Confidential Secretary To Police Chief
5609                             Assistant To The Director
6544                             Assistant To The Director
6594     Confidential Secretary To Chief Operating Officer
6832                       Confidential Secretary To Mayor
11028                      Confidential Secretary To Mayor
Name: Job Title, dtype: object

We can look for other filler words too, like `'the'` and `'for'`.

In [72]:
jobtitles[jobtitles.str.lower().str.contains(r'\bthe\b', regex=True)]

1638    Assistant To The Chief Operating Officer
2238                   Assistant To The Director
5609                   Assistant To The Director
6544                   Assistant To The Director
Name: Job Title, dtype: object

In [73]:
jobtitles[jobtitles.str.lower().str.contains(r'\bfor\b', regex=True)]

3449     Assistant For Community Outreach
6889     Assistant For Community Outreach
10810    Assistant For Community Outreach
Name: Job Title, dtype: object

We should probably remove these "glue" words.

### Roman numerals (e.g. "Ii")

Lastly, let's try and identify job titles that have roman numerals at the end, like `'i'` (1), `'ii'` (2), `'iii'` (3), or `'iv'` (4). As before, we'll convert to lowercase first.

In [74]:
jobtitles[jobtitles.str.lower().str.contains(r'\bi+v?\b', regex=True)]

5                   Police Officer Ii
10                  Police Officer Ii
48       Fire Prevention Inspector Ii
53                  Police Officer Ii
56                  Police Officer Ii
                     ...             
12820                     Lifeguard I
12821                   Pool Guard Ii
12822           Clerical Assistant Ii
12828               Police Officer Ii
12830                Police Officer I
Name: Job Title, Length: 6087, dtype: object

Let's get rid of those numbers, too.

### Fixing punctuation and removing "glue" words and roman numerals

Let's put the preceeding three steps together and canonicalize job titles by:
- converting to lowercase,
- removing each occurrence of `'to'`, `'the'`, and `'for'`,
- replacing each non-letter/digit/space character with a space,
- replacing each sequence of roman numerals – either `'i'`, `'ii'`, `'iii'`, or `'iv'` at the end with nothing, and
- replacing each sequence of multiple spaces with a single space.

In [75]:
jobtitles = (
    jobtitles
    .str.lower()
    .str.replace(r'\bto\b|\bthe\b|\bfor\b', '', regex=True)
    .str.replace(r'[^A-Za-z0-9 ]', ' ', regex=True)
    .str.replace(r'\bi+v?\b', '', regex=True)
    .str.replace(r' +', ' ', regex=True)               # ' +' matches 1 or more occurrences of a space.
    .str.strip()                                       # Removes leading/trailing spaces if present.
)

In [76]:
jobtitles.sample(5)

7387            administrative aide
11401             library assistant
780      fire prevention supervisor
6772                     dispatcher
12459                   criminalist
Name: Job Title, dtype: object

In [77]:
(jobtitles == 'police officer').sum()

np.int64(1378)

### Possible issue: inconsistent representations

Another possible issue is that some job titles may have inconsistent representations of the same word (e.g. `'Asst.'` vs `'Assistant'`).

In [78]:
jobtitles[jobtitles.str.contains('asst')].value_counts()

Series([], Name: count, dtype: int64)

In [79]:
jobtitles[jobtitles.str.contains('assistant')].value_counts().head()

Job Title
library assistant            385
assistant engineer civil     297
clerical assistant           111
assistant center director     50
assistant chemist             45
Name: count, dtype: int64

The 2020 salaries dataset had several of these issues, but fortunately they appear to be fixed for us in the 2022 dataset (thanks, Transparent California).

## Bag of words 💰

### Text similarity

Our idea is to measure the similarity of two job titles by counting the number of shared words between the job titles. How do we actually do that, for all of the job titles we have?

### A counts matrix

Let's create a "counts" matrix, such that:
- there is 1 row per job title,
- there is 1 column per **unique** word that is used in job titles, and
- the value in row `title` and column `word` is the number of occurrences of `word` in `title`.

Such a matrix might look like:

| | senior | lecturer | teaching | professor | assistant | associate |
| --- | --- | --- | --- | --- | --- | --- |
| **senior lecturer** | 1 | 1 | 0 | 0 | 0 | 0 |
| **assistant teaching professor** | 0 | 0 | 1 | 1 | 1 | 0 | 
| **associate professor** | 0 | 0 | 0 | 1 | 0 | 1 |
| **senior assistant to the assistant professor** | 1 | 0 | 0 | 1 | 2 | 0 |

Then, we can make statements like: "assistant teaching professor" is more similar to "associate professor" than to "senior lecturer".

### Creating a counts matrix

First, we need to determine all words that are used across all job titles.

In [80]:
jobtitles.str.split()

0                              [city, attorney]
1                                       [mayor]
2                    [assistant, police, chief]
3                            [police, sergeant]
4        [assistant, retirement, administrator]
                          ...                  
12826              [word, processing, operator]
12827                                 [cashier]
12828                         [police, officer]
12829                      [police, dispatcher]
12830                         [police, officer]
Name: Job Title, Length: 12831, dtype: object

In [81]:
# The .explode method concats the lists together.
all_words = jobtitles.str.split().explode()
all_words

0              city
0          attorney
1             mayor
2         assistant
2            police
            ...    
12828       officer
12829        police
12829    dispatcher
12830        police
12830       officer
Name: Job Title, Length: 30057, dtype: object

Next, to determine the columns of our matrix, we need to find a list of all **unique** words used in titles. We can do this with `np.unique`, but `value_counts` shows us the distribution, which is interesting.

In [82]:
unique_words = all_words.value_counts()
unique_words

Job Title
police       2299
officer      1608
assistant    1267
fire         1109
engineer     1083
             ... 
stores          1
emergency       1
relations       1
dna             1
gardener        1
Name: count, Length: 330, dtype: int64

Note that in `unique_words.index`, job titles are sorted by number of occurrences!

For each of the 330 unique words that are used in job titles, we can count the number of occurrences of the word in each job title.
- `'deputy fire chief'` contains the word `'deputy'` once, the word `'fire'` once, and the word `'chief'` once.
- `'assistant managers assistant'` contains the word `'assistant'` twice and the word `'managers'` once.

In [83]:
# Created using a dictionary to avoid a "DataFrame is highly fragmented" warning.
counts_dict = {}
for word in unique_words.index:
    re_pat = fr'\b{word}\b'
    counts_dict[word] = jobtitles.str.count(re_pat).astype(int).tolist()
    
counts_df = pd.DataFrame(counts_dict).set_index(jobtitles)

In [84]:
counts_df.head()

,police,officer,assistant,fire,engineer,civil,technician,associate,senior,worker,...,medical,lakes,technical,operators,governmental,stores,emergency,relations,dna,gardener
Job Title,,,,,,,,,,,,,,,,,,,,,
city attorney,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
mayor,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
assistant police chief,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
police sergeant,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
assistant retirement administrator,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [85]:
counts_df.shape

(12831, 330)

`counts_df` has one row for all 12831 employees, and one column for each unique word that is used in a job title.

Its third row, for example, tells us that the third job title contains `'police'` once and `'assistant'` once.

### Bag of words

- The **bag of words** model represents texts (e.g. job titles, sentences, documents) as **vectors of word counts**.
    - The "counts" matrices we have worked with so far were created using the bag of words model.
    - The bag of words model defines a **vector space** in $\mathbb{R}^{\text{number of unique words}}$.
    - In the matrix on the previous slide, each **row** was a vector corresponding to a specific job title.
- It is called "bag of words" because it doesn't consider **order**.

<center><img src='imgs/bag-of-words.jpeg' width=45%></center>

<center><a href="https://42f6861cgkip12ijm63i3orf-wpengine.netdna-ssl.com/wp-content/uploads/2020/12/2020-07-bagofwords.jpg">(source)</a></center>

## Cosine similarity

### Question: What job titles are most similar to `'deputy fire chief'`?

- We want to quantify similarity by comparing word counts between titles.
- `counts_df` contains a vector of word counts for each title.
- How can we use these vectors to measure similarity?

### Counting shared words

To start, let's compare the row vectors for `'deputy fire chief'` and `'fire battalion chief'`.

In [86]:
dfc = counts_df.loc['deputy fire chief'].iloc[0]
dfc

police       0
officer      0
assistant    0
fire         1
engineer     0
            ..
stores       0
emergency    0
relations    0
dna          0
gardener     0
Name: deputy fire chief, Length: 330, dtype: int64

In [87]:
fbc = counts_df.loc['fire battalion chief'].iloc[0]
fbc

police       0
officer      0
assistant    0
fire         1
engineer     0
            ..
stores       0
emergency    0
relations    0
dna          0
gardener     0
Name: fire battalion chief, Length: 330, dtype: int64

We can stack these two vectors horizontally.

In [88]:
pair_counts = (
    pd.concat([dfc, fbc], axis=1)
    .sort_values(by=['deputy fire chief', 'fire battalion chief'], ascending=False)
    .head(10)
    .T
)

pair_counts

,fire,chief,deputy,battalion,police,officer,assistant,engineer,civil,technician
deputy fire chief,1,1,1,0,0,0,0,0,0,0
fire battalion chief,1,1,0,1,0,0,0,0,0,0


`'deputy fire chief'` and `'fire battalion chief'` have 2 shared words in common. One way to arrive at this result mathematically is by taking their **dot product**:

In [89]:
np.dot(pair_counts.iloc[0], pair_counts.iloc[1])

np.int64(2)

### The dot product

- If $\vec{a} = \begin{bmatrix} a_1 & a_2 & ... & a_n \end{bmatrix}^T$ and $\vec{b} = \begin{bmatrix} b_1 & b_2 & ... & b_n \end{bmatrix}^T$ are two vectors, then their **dot product** $\vec{a} \cdot \vec{b}$ is defined as:

$$\vec{a} \cdot \vec{b} = a_1b_1 + a_2b_2 + ... + a_nb_n$$

- The dot product also has a **geometric** interpretation. If $|\vec{a}|$ and $|\vec{b}|$ are the $L_2$ norms (lengths) of $\vec{a}$ and $\vec{b}$, and $\theta$ is the angle between $\vec{a}$ and $\vec{b}$, then:

$$\vec{a} \cdot \vec{b} = |\vec{a}| |\vec{b}| \cos \theta$$

<center><img src='imgs/dot-prod.png' width=20%>(<a href="https://byjus.com/physics/scalar-and-vector-products/">source</a>)</center>

- $\cos \theta$ is equal to its maximum value (1) when $\theta = 0$, i.e. when $\vec{a}$ and $\vec{b}$ point in the same direction. 

### Cosine similarity and bag of words

To measure the similarity between two word vectors, instead of just counting the number of shared words, we should compute their **normalized** dot product, also known as their **cosine similarity**.

$$\cos \theta = \boxed{\frac{\vec{a} \cdot \vec{b}}{|\vec{a}| | \vec{b}|}}$$

- If all elements in $\vec{a}$ and $\vec{b}$ are non-negative, then $\cos \theta$ ranges from 0 to 1.
- 🚨 **Key idea: The larger $\cos \theta$ is, the more similar the two vectors are!**
- **It is important to normalize by the lengths of the vectors**, otherwise texts with more words will have artificially high similarities with other texts.

### Normalizing

$$\cos \theta = \boxed{\frac{\vec{a} \cdot \vec{b}}{|\vec{a}| | \vec{b}|}}$$
- Why can't we just use the dot product – that is, why must we divide by $|\vec{a}| | \vec{b}|$?

- Consider the following example:

| | big | data | science | 
| --- | --- | --- | --- |
| **big big big big data** | 4 | 1 | 0 |
| **big data science** | 1 | 1 | 1 |
| **science big data** | 1 | 1 | 1 |

| Pair | Dot Product | Cosine Similarity |
| --- | --- | --- |
| **big data science** and **big big big big data** | 5 | 0.7001 |
| **big data science** and **science big data** | 3 | 1 |

- `'big big big big data'` has a large dot product with `'big data science'` just because it has the word `'big'` four times. But intuitively, `'big data science'` and `'science big data'` should be as similar as possible, since they're permutations of the same sentence.
- **So, make sure to compute the cosine similarity – don't just use the dot product!**

**Note**: Sometimes, you will see the **cosine distance** being used. It is the complement of cosine similarity:
  
  $$\text{dist}(\vec{a}, \vec{b}) = 1 - \cos \theta$$
  
If $\text{dist}(\vec{a}, \vec{b})$ is small, the two word vectors are similar.

### A recipe for computing similarities

Given a set of documents, to find the **most similar** text to one _document_ $d$ in particular:
- Use the bag of words model to create a counts matrix, in which:
    - there is 1 row per document,
    - there is 1 column per **unique** word that is used across documents, and
    - the value in row `doc` and column `word` is the number of occurrences of `word` in `doc`.
- Compute the cosine similarity between $d$'s row vector and all other documents' row vectors.
- The other document with the greatest cosine similarity is the most similar, under the bag of words model.

### Example: Global warming 🌎

Consider the following three documents.

In [90]:
sentences = pd.Series([
    'I really really want global peace',
    'I must enjoy global warming',
    'We must solve climate change'
])

sentences

0    I really really want global peace
1          I must enjoy global warming
2         We must solve climate change
dtype: object

Let's represent each document using the bag of words model.

In [91]:
unique_words = sentences.str.split().explode().value_counts()
unique_words

I          2
really     2
global     2
must       2
want       1
peace      1
enjoy      1
warming    1
We         1
solve      1
climate    1
change     1
Name: count, dtype: int64

In [92]:
counts_dict = {}
for word in unique_words.index:
    re_pat = fr'\b{word}\b'
    counts_dict[word] = sentences.str.count(re_pat).astype(int).tolist()
    
counts_df = pd.DataFrame(counts_dict).set_index(sentences)

In [93]:
counts_df

,I,really,global,must,want,peace,enjoy,warming,We,solve,climate,change
I really really want global peace,1,2,1,0,1,1,0,0,0,0,0,0
I must enjoy global warming,1,0,1,1,0,0,1,1,0,0,0,0
We must solve climate change,0,0,0,1,0,0,0,0,1,1,1,1


Let's now find the cosine similarity between each document.

In [94]:
counts_df

,I,really,global,must,want,peace,enjoy,warming,We,solve,climate,change
I really really want global peace,1,2,1,0,1,1,0,0,0,0,0,0
I must enjoy global warming,1,0,1,1,0,0,1,1,0,0,0,0
We must solve climate change,0,0,0,1,0,0,0,0,1,1,1,1


In [95]:
def sim_pair(s1, s2):
    return np.dot(s1, s2) / (np.linalg.norm(s1) * np.linalg.norm(s2))

In [96]:
# Look at the documentation of the .corr method to see how this works!
counts_df.T.corr(sim_pair)

,I really really want global peace,I must enjoy global warming,We must solve climate change
I really really want global peace,1.000000,0.316228,0.0
I must enjoy global warming,0.316228,1.000000,0.2
We must solve climate change,0.000000,0.200000,1.0


**Issue**: Bag of words only encodes the **words** that each document uses, not their **meanings**.
- "I really really want global peace" and "We must solve climate change" have similar meanings, but have no shared words, and thus a low cosine similarity.
- "I really really want global peace" and "I must enjoy global warming" have very different meanings, but a relatively high cosine similarity.

### Pitfalls of the bag of words model

Remember, the key assumption underlying the bag of words model is that **two documents are similar if they share many words in common**.

- The bag of words model doesn't consider **order**.
    - The job titles `'deputy fire chief'` and `'chief fire deputy'` are treated as the same.

- The bag of words model doesn't consider the **meaning** of words.
    - `'I love data science'` and `'I hate data science'` share 75% of their words, but have very different meanings.

- The bag of words model treats all words as being equally important.
    - `'deputy'` and `'fire'` have the same importance, even though `'fire'` is probably more important in describing someone's job title.
    - <span style="color:red"><b>Let's address this point.</b></span>

### Question 🤔
        
Consider the following corpus:

```
Document number Content
1               yesterday rainy today sunny
2               yesterday sunny today sunny
3               today rainy yesterday today
4               yesterday yesterday today today
```
        
1. Using a bag-of-words representation, which two documents have the largest dot product?
2. Using a bag-of-words representation, what is the cosine similarity between documents 2 and 3?

## TF-IDF

### The importance of words

**Issue**: The bag of words model doesn't know which words are "important" in a document. Consider the following document:

<center>"my brother has a friend named <b>billy</b> who has an uncle named <b>billy</b>"</center>

How do we determine which words are important?
- The most common words ("the", "has") often **don't** have much meaning!
- The very rare words are also less important!

**Goal**: Find a way of quantifying the importance of a word in a document by **balancing the above two factors**, i.e. find the word that **best summarizes** a document.

### Term frequency

- The **term frequency** of a word (term) $t$ in a document $d$, denoted $\text{tf}(t, d)$, is the proportion of words in document $d$ that are equal to $t$.

$$
\text{tf}(t, d) = \frac{\text{\# of occurrences of } t \text{ in } d}{\text{total \# of words in } d}
$$


- **Example**: What is the term frequency of "billy" in the following document?

<center>"my brother has a friend named <b>billy</b> who has an uncle named <b>billy</b>"</center>

- **Answer**: $\frac{2}{13}$.

- Intuition: Words that occur often within a document are important to the document's meaning.
    - If $\text{tf}(t, d)$ is large, then word $t$ occurs often in $d$.
    - If $\text{tf}(t, d)$ is small, then word $t$ does not occur often $d$.

- Issue: "has" also has a TF of $\frac{2}{13}$, but it seems less important than "billy".

### Inverse document frequency

- The **inverse document frequency** of a word $t$ in a collection of documents $d_1, d_2, \ldots$ is:

$$
\text{idf}(t) = \log \left( \frac{\text{total \# of documents}}{\text{\# of documents in which } t \text{ appears}} \right)
$$


- **Example**: What is the inverse document frequency of "billy" in the following three documents?
    - "my brother has a friend named **billy** who has an uncle named **billy**"
    - "my favorite artist is named jilly boel"
    - "why does he talk about someone named **billy** so often"

- **Answer**: $\log \left(\frac{3}{2}\right) \approx 0.4055$.

- Intuition: If a word appears in every document (like "the" or "has"), it is probably not a good summary of any one document.
    - If $\text{idf}(t)$ is large, then $t$ is rarely found in documents.
    - If $\text{idf}(t)$ is small, then $t$ is commonly found in documents.
    - Think of $\text{idf}(t)$ as the "rarity factor" of $t$ across documents – the larger $\text{idf}(t)$ is, the more rare $t$ is.

### Intuition

$$
\text{tf}(t, d) = \frac{\text{\# of occurrences of } t \text{ in } d}{\text{total \# of words in } d}
$$

$$
\text{idf}(t) = \log \left( \frac{\text{total \# of documents}}{\text{\# of documents in which } t \text{ appears}} \right)
$$

**Goal:** Quantify how well the word $t$ **summarizes** document $d$.


- If $\text{tf}(t, d)$ is small, then $t$ doesn't occur very often in $d$, so $t$ can't be a good summary of $d$.

- If $\text{idf}(t)$ is small, then $t$ occurs often amongst all documents, and so it is not a good summary of any one document.

- If $\text{tf}(t, d)$ and $\text{idf}(t)$ are both large, then **$t$ occurs often in $d$ but rarely overall**. This makes $t$ **a good summary** of document $d$.

### Term Frequency–Inverse Document Frequency

The **term frequency–inverse document frequency (TF–IDF)** of a word $t$ in a document $d$ is the product:

$$
\begin{align*}
\text{tfidf}(t, d)
&= \text{tf}(t, d) \cdot \text{idf}(t) \\
&= 
\frac{\text{\# of occurrences of } t \text{ in } d}{\text{total \# of words in } d}
\cdot 
\log \left(
\frac{\text{total \# of documents}}{\text{\# of documents in which } t \text{ appears}}
\right)
\end{align*}
$$


- If $\text{tfidf}(t, d)$ is large, then $t$ is a good summary of $d$, because $t$ occurs often in $d$ but rarely across all documents.

- TF-IDF is a **heuristic** – it has no probabilistic justification.

- To know if $\text{tfidf}(t, d)$ is large for one particular word $t$, we need to compare it to $\text{tfidf}(t_i, d)$, for several different words $t_i$.

### Computing TF-IDF

**Question**: What is the TF-IDF of "global" in the second sentence?

In [97]:
sentences

0    I really really want global peace
1          I must enjoy global warming
2         We must solve climate change
dtype: object

**Answer**:

In [98]:
tf = sentences.iloc[1].count('global') / len(sentences.iloc[1].split())
tf

0.2

In [99]:
idf = np.log(len(sentences) / sentences.str.contains('global').sum())
idf

np.float64(0.4054651081081644)

In [100]:
tf * idf

np.float64(0.08109302162163289)

**Question**: Is this big or small? Is "global" the **best** summary of the second sentence?

### TF-IDF of all words in all documents

On its own, the TF-IDF of a word in a document doesn't really tell us anything; we must compare it to TF-IDFs of other words in that same document.

In [101]:
sentences

0    I really really want global peace
1          I must enjoy global warming
2         We must solve climate change
dtype: object

In [102]:
unique_words = np.unique(sentences.str.split().explode())
unique_words

array(['I', 'We', 'change', 'climate', 'enjoy', 'global', 'must', 'peace',
       'really', 'solve', 'want', 'warming'], dtype=object)

In [103]:
tfidf_dict = {}

for word in unique_words:
    re_pat = fr'\b{word}\b'
    tf = sentences.str.count(re_pat) / sentences.str.split().str.len()
    idf = np.log(len(sentences) / sentences.str.contains(re_pat).sum())
    tfidf_dict[word] = tf * idf
    
tfidf = pd.DataFrame(tfidf_dict).set_index(sentences)

In [104]:
tfidf

,I,We,change,climate,enjoy,global,must,peace,really,solve,want,warming
I really really want global peace,0.067578,0.000000,0.000000,0.000000,0.000000,0.067578,0.000000,0.183102,0.366204,0.000000,0.183102,0.000000
I must enjoy global warming,0.081093,0.000000,0.000000,0.000000,0.219722,0.081093,0.081093,0.000000,0.000000,0.000000,0.000000,0.219722
We must solve climate change,0.000000,0.219722,0.219722,0.219722,0.000000,0.000000,0.081093,0.000000,0.000000,0.219722,0.000000,0.000000


### Interpreting TF-IDFs

In [105]:
display_df(tfidf, cols=12)

,I,We,change,climate,enjoy,global,must,peace,really,solve,want,warming
I really really want global peace,0.067578,0.000000,0.000000,0.000000,0.000000,0.067578,0.000000,0.183102,0.366204,0.000000,0.183102,0.000000
I must enjoy global warming,0.081093,0.000000,0.000000,0.000000,0.219722,0.081093,0.081093,0.000000,0.000000,0.000000,0.000000,0.219722
We must solve climate change,0.000000,0.219722,0.219722,0.219722,0.000000,0.000000,0.081093,0.000000,0.000000,0.219722,0.000000,0.000000


The above DataFrame tells us that:
- the TF-IDF of `'really'` in the first sentence is $\approx$ 0.37,
- the TF-IDF of `'climate'` in the second sentence is 0.

Note that there are two ways that $\text{tfidf}(t, d) = \text{tf}(t, d) \cdot \text{idf}(t)$ can be 0:

- If $t$ appears in **every document**, then  
  $\text{idf}(t) = \log \left( \frac{\text{\# of documents}}{\text{\# of documents}} \right) = \log(1) = 0$.

- If $t$ does **not appear in document** $d$, then  
  $\text{tf}(t, d) = \frac{0}{\text{len}(d)} = 0$.


The word that **best summarizes** a document is the word with the highest TF-IDF for that document:

In [106]:
tfidf.idxmax(axis=1)

I really really want global peace    really
I must enjoy global warming           enjoy
We must solve climate change             We
dtype: object

Look closely at the rows of `tfidf` – in documents 2 and 3, the max TF-IDF is not unique!

## Example: State of the Union addresses 🎤

### State of the Union addresses

The 2023 State of the Union address was on February 7th, 2023.

### The data

In [107]:
sotu_txt = Path('data') / 'stateoftheunion1790-2023.txt'
sotu = sotu_txt.read_text()

In [108]:
len(sotu)

10577941

The entire **corpus** (another word for "set of documents") is over 10 million characters long... let's not display it in our notebook.

In [109]:
print(sotu[:1600])

The Project Gutenberg EBook of Complete State of the Union Addresses,
from 1790 to the Present. Speeches beginning in 2002 are from UCSB The American Presidency Project.
Speeches from 2018-2023 were manually downloaded from whitehouse.gov.

Character set encoding: UTF8

The addresses are separated by three asterisks


CONTENTS

  George Washington, State of the Union Address, January 8, 1790
  George Washington, State of the Union Address, December 8, 1790
  George Washington, State of the Union Address, October 25, 1791
  George Washington, State of the Union Address, November 6, 1792
  George Washington, State of the Union Address, December 3, 1793
  George Washington, State of the Union Address, November 19, 1794
  George Washington, State of the Union Address, December 8, 1795
  George Washington, State of the Union Address, December 7, 1796
  John Adams, State of the Union Address, November 22, 1797
  John Adams, State of the Union Address, December 8, 1798
  John Adams, State of 

Each speech is separated by `'***'`.

In [110]:
speeches = sotu.split('\n***\n')[1:]

In [111]:
len(speeches)

233

Note that each "speech" currently contains other information, like the name of the president and the date of the address.

In [112]:
print(speeches[-1][:1000])


State of the Union Address
Joseph R. Biden Jr.  
February 7, 2023

  Mr. Speaker. Madam Vice President. Our First Lady and Second
Gentleman. Members of Congress and the Cabinet. Leaders of our
military. Mr. Chief Justice, Associate Justices, and retired Justices
of the Supreme Court. And you, my fellow Americans.
  I start tonight by congratulating the members of the 118th Congress
and the new Speaker of the House, Kevin McCarthy. Mr. Speaker, I look
forward to working together.
  I also want to congratulate the new leader of the House Democrats and
the first Black House Minority Leader in history, Hakeem Jeffries.
  Congratulations to the longest serving Senate leader in history,
Mitch McConnell.
  And congratulations to Chuck Schumer for another term as Senate
Majority Leader, this time with an even bigger majority.
  And I want to give special recognition to someone who I think will be
considered the greatest Speaker in the history of this country, Nancy
Pelosi.
  The story of Amer

Let's extract just the speech text.

In [113]:
def extract_struct(speech):
    L = speech.strip().split('\n', maxsplit=3)
    L[3] = re.sub(r"[^A-Za-z' ]", ' ', L[3]).lower()
    return dict(zip(['speech', 'president', 'date', 'contents'], L))

In [114]:
speeches_df = pd.DataFrame(list(map(extract_struct, speeches)))
speeches_df

,speech,president,date,contents
0,State of the Union Address,George Washington,"January 8, 1790",fellow citizens of the senate and house of re...
1,State of the Union Address,George Washington,"December 8, 1790",fellow citizens of the senate and house of re...
2,State of the Union Address,George Washington,"October 25, 1791",fellow citizens of the senate and house of re...
3,State of the Union Address,George Washington,"November 6, 1792",fellow citizens of the senate and house of re...
4,State of the Union Address,George Washington,"December 3, 1793",fellow citizens of the senate and house of re...
...,...,...,...,...
228,State of the Union Address,Donald J. Trump,"February 5, 2019",madam speaker mr vice president members of...
229,State of the Union Address,Donald J. Trump,"February 4, 2020",thank you very much thank you thank you ver...
230,State of the Union Address,Joseph R. Biden Jr.,"April 28, 2021",thank you thank you thank you good to be b...
231,State of the Union Address,Joseph R. Biden Jr.,"March 1, 2022",madam speaker madam vice president and our ...


### Finding the most important words in each speech

Here, a "document" is a speech. We have 233 documents.

In [115]:
speeches_df

,speech,president,date,contents
0,State of the Union Address,George Washington,"January 8, 1790",fellow citizens of the senate and house of re...
1,State of the Union Address,George Washington,"December 8, 1790",fellow citizens of the senate and house of re...
2,State of the Union Address,George Washington,"October 25, 1791",fellow citizens of the senate and house of re...
3,State of the Union Address,George Washington,"November 6, 1792",fellow citizens of the senate and house of re...
4,State of the Union Address,George Washington,"December 3, 1793",fellow citizens of the senate and house of re...
...,...,...,...,...
228,State of the Union Address,Donald J. Trump,"February 5, 2019",madam speaker mr vice president members of...
229,State of the Union Address,Donald J. Trump,"February 4, 2020",thank you very much thank you thank you ver...
230,State of the Union Address,Joseph R. Biden Jr.,"April 28, 2021",thank you thank you thank you good to be b...
231,State of the Union Address,Joseph R. Biden Jr.,"March 1, 2022",madam speaker madam vice president and our ...


A rough sketch of what we'll compute:

```
for each word t:
    for each speech d:
        compute tfidf(t, d)
```

In [116]:
unique_words = speeches_df['contents'].str.split().explode().value_counts()
# Take the top 500 most common words for speed
unique_words = unique_words.iloc[:500].index
unique_words

Index(['the', 'of', 'to', 'and', 'in', 'a', 'that', 'for', 'be', 'our',
       ...
       'desire', 'call', 'submitted', 'increasing', 'months', 'point', 'trust',
       'throughout', 'set', 'object'],
      dtype='object', name='contents', length=500)

### 💡 Pro-Tip: Using `tqdm`

This code takes a while to run, so we'll use the `tdqm` package to track its progress. (Install with `mamba install tqdm` if needed).

In [117]:
from tqdm.notebook import tqdm

tfidf_dict = {}
tf_denom = speeches_df['contents'].str.split().str.len()

# Wrap the sequence with `tqdm()` to display a progress bar
for word in tqdm(unique_words):
    re_pat = fr' {word} ' # Imperfect pattern for speed.
    tf = speeches_df['contents'].str.count(re_pat) / tf_denom
    idf = np.log(len(speeches_df) / speeches_df['contents'].str.contains(re_pat).sum())
    tfidf_dict[word] =  tf * idf

  0%|          | 0/500 [00:00<?, ?it/s]

In [118]:
tfidf = pd.DataFrame(tfidf_dict)
tfidf

,the,of,to,and,in,a,that,for,be,our,...,desire,call,submitted,increasing,months,point,trust,throughout,set,object
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000328,0.000000,0.000380,0.000000,0.000000,0.000429,0.000000,0.000000,0.002043
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000255,0.000438,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001057
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000267,0.000180,0.000000,0.000000,0.000406,0.000000,0.000348,0.000644
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000286,0.000171,0.000000,0.000000,0.000000,0.000000,0.000670,0.000217,0.000000,0.000709
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000610,0.000000,0.000000,0.000000,0.000000,0.000000,0.000238,0.000462,0.000000,0.000377
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000318,0.000069,0.000000,0.000000,0.000214,0.000000
229,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000171,0.000000,0.000000,0.000356,0.000062,0.000000,0.000072,0.000064,0.000000
230,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000052,0.000056,0.000147,0.000059,0.000171,0.000101,0.000000
231,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000222,0.000000,0.000051,0.000166,0.000145,0.000174,0.000113,0.000149,0.000000


Note that the TF-IDFs of many common words are all 0!

### Summarizing speeches

By using `idxmax`, we can find the word with the highest TF-IDF in each speech.

In [119]:
summaries = tfidf.idxmax(axis=1)
summaries

0          object
1      convention
2       provision
3       provision
4           ought
          ...    
228       tonight
229       tonight
230          it's
231       tonight
232          it's
Length: 233, dtype: object

What if we want to see the 5 words with the highest TF-IDFs, for each speech?

In [120]:
def five_largest(row):
    return ', '.join(row.index[row.argsort()][-5:])

In [121]:
keywords = tfidf.apply(five_largest, axis=1)
keywords_df = pd.concat([
    speeches_df['president'],
    speeches_df['date'],
    keywords
], axis=1)

In [122]:
keywords_df

,president,date,0
0,George Washington,"January 8, 1790","your, proper, regard, ought, object"
1,George Washington,"December 8, 1790","case, established, object, commerce, convention"
2,George Washington,"October 25, 1791","community, upon, lands, proper, provision"
3,George Washington,"November 6, 1792","subject, upon, information, proper, provision"
4,George Washington,"December 3, 1793","having, vessels, executive, shall, ought"
...,...,...,...
228,Donald J. Trump,"February 5, 2019","get, jobs, americans, it's, tonight"
229,Donald J. Trump,"February 4, 2020","jobs, it's, americans, percent, tonight"
230,Joseph R. Biden Jr.,"April 28, 2021","get, americans, percent, jobs, it's"
231,Joseph R. Biden Jr.,"March 1, 2022","let, jobs, americans, get, tonight"


### Summary

- Text can be represented numerically by counting word occurrences — the **bag-of-words** approach.  
- The **TF–IDF** statistic refines this idea by weighting words based on both  
  - how often they appear in a document (**term frequency**, $\text{tf}(t, d)$), and  
  - how rare they are across all documents (**inverse document frequency**, $\text{idf}(t)$).  
- Words with the highest TF–IDF values are often the most distinctive or representative of a document’s content.


---
### Acknowledgment
Adapted from course materials for DSC 80: The Practice and Application of Data Science
 (UC San Diego).